### Similarity computations

In [8]:
import numpy as np

# Sample embeddings (2D for visualization, real = 768D)
embedding_A = np.array([3.0, 4.0])  # "machine learning"
embedding_B = np.array([3.5, 4.5])  # "deep learning"
embedding_C = np.array([1.0, 0.0])  # "pizza recipe"

In [9]:
# 1. Cosine Similarity
def cosine_similarity(a, b):
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)  # sqrt(sum of squares)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

print(f"Cosine(A, B): {cosine_similarity(embedding_A, embedding_B):.4f}")
print(f"Cosine(A, C): {cosine_similarity(embedding_A, embedding_C):.4f}")

Cosine(A, B): 0.9998
Cosine(A, C): 0.6000


In [10]:
# 2. Dot product  for embeddings
print(f"\nDot(A, B): {np.dot(embedding_A, embedding_B):.4f}")
print(f"Dot(A, C): {np.dot(embedding_A, embedding_C):.4f}")


Dot(A, B): 28.5000
Dot(A, C): 3.0000


In [11]:
# Efficient: Normalize then Dot Product
def normalize(vec):
    return vec / np.linalg.norm(vec)

embedding_A_norm = normalize(embedding_A)
embedding_B_norm = normalize(embedding_B)
embedding_C_norm = normalize(embedding_C)

# Now dot product = cosine similarity
print(f"Dot(A_norm, B_norm): {np.dot(embedding_A_norm, embedding_B_norm):.4f}")
print(f"Dot(A_norm, C_norm): {np.dot(embedding_A_norm, embedding_C_norm):.4f}") # Added Dot(A_norm, C_norm)

Dot(A_norm, B_norm): 0.9998
Dot(A_norm, C_norm): 0.6000


In [13]:
# 4. L2 Distance (Euclidean)
def l2_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

print(f"L2(A, B): {l2_distance(embedding_A, embedding_B):.4f}")  # Small = similar
print(f"L2(A, C): {l2_distance(embedding_A, embedding_C):.4f}")    # Large = different

L2(A, B): 0.7071
L2(A, C): 4.4721


### Batch computation for many vectors

In [14]:
embeddings = np.array([embedding_A, embedding_B, embedding_C])
embeddings_normalized = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

# Compute all pairwise similarities at once (matrix multiplication)
similarity_matrix = np.dot(embeddings_normalized, embeddings_normalized.T)
print("\nSimilarity Matrix:")
print(similarity_matrix)


Similarity Matrix:
[[1.         0.99984614 0.6       ]
 [0.99984614 1.         0.61394061]
 [0.6        0.61394061 1.        ]]


### L2 distance relates to cosine

In [16]:
# For normalized vectors, L2 distance relates to cosine:
# L2² = 2(1 - cosine_similarity)

# Using normalized embeddings A and B
l2_squared_A_B_norm = l2_distance(embedding_A_norm, embedding_B_norm)**2
cosine_A_B_norm = np.dot(embedding_A_norm, embedding_B_norm)
print(f"L2²(A_norm, B_norm): {l2_squared_A_B_norm:.4f}")
print(f"2 * (1 - Cosine(A_norm, B_norm)): {2 * (1 - cosine_A_B_norm):.4f}")

# Using normalized embeddings A and C
l2_squared_A_C_norm = l2_distance(embedding_A_norm, embedding_C_norm)**2
cosine_A_C_norm = np.dot(embedding_A_norm, embedding_C_norm)
print(f"\nL2²(A_norm, C_norm): {l2_squared_A_C_norm:.4f}")
print(f"2 * (1 - Cosine(A_norm, C_norm)): {2 * (1 - cosine_A_C_norm):.4f}")

L2²(A_norm, B_norm): 0.0003
2 * (1 - Cosine(A_norm, B_norm)): 0.0003

L2²(A_norm, C_norm): 0.8000
2 * (1 - Cosine(A_norm, C_norm)): 0.8000


### Hybrid Search Implementation

In [2]:
%%capture
!pip install rank-bm25

In [3]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import numpy as np

In [4]:
class HybridSearchSystem:
    def __init__(self, documents):
        self.documents = documents

        # Initialize dense retrieval
        print("Loading bi-encoder...")
        self.bi_encoder = SentenceTransformer('all-mpnet-base-v2')
        self.doc_embeddings = self.bi_encoder.encode(documents, show_progress_bar=True)

        # Normalize for fast dot product
        self.doc_embeddings = self.doc_embeddings / np.linalg.norm(
            self.doc_embeddings, axis=1, keepdims=True
        )

        # Initialize sparse retrieval
        print("Building BM25 index...")
        tokenized_docs = [doc.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs, k1=1.5, b=0.75)

        # Initialize re-ranker (optional)
        print("Loading cross-encoder...")
        self.cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')

    def search(self, query, top_k=10, use_rerank=True):
        """
        Three-stage hybrid search
        """
        # Stage 1: Dense retrieval
        query_embedding = self.bi_encoder.encode([query])[0]
        query_embedding = query_embedding / np.linalg.norm(query_embedding)

        dense_scores = np.dot(self.doc_embeddings, query_embedding)
        dense_ranks = np.argsort(-dense_scores)  # Descending order

        # Stage 2: Sparse retrieval (BM25)
        tokenized_query = query.lower().split()
        bm25_scores = self.bm25.get_scores(tokenized_query)
        sparse_ranks = np.argsort(-bm25_scores)

        # Stage 3: RRF Fusion
        rrf_scores = self._reciprocal_rank_fusion(
            dense_ranks[:100],  # Top 100 from dense
            sparse_ranks[:100], # Top 100 from sparse
            k=60
        )

        # Get top 20 for re-ranking
        top_20_indices = np.argsort(-rrf_scores)[:20]

        if use_rerank:
            # Stage 4: Re-rank with cross-encoder
            pairs = [[query, self.documents[i]] for i in top_20_indices]
            rerank_scores = self.cross_encoder.predict(pairs)

            # Sort by rerank scores
            sorted_indices = np.argsort(-rerank_scores)
            final_indices = top_20_indices[sorted_indices[:top_k]]
            final_scores = rerank_scores[sorted_indices[:top_k]]
        else:
            # Just use RRF scores
            final_indices = top_20_indices[:top_k]
            final_scores = rrf_scores[final_indices]

        # Return results
        results = []
        for idx, score in zip(final_indices, final_scores):
            results.append({
                'document': self.documents[idx],
                'score': float(score),
                'index': int(idx)
            })

        return results

    def _reciprocal_rank_fusion(self, rank_list1, rank_list2, k=60):
        """
        Combine two ranking lists using RRF
        """
        scores = np.zeros(len(self.documents))

        # Add scores from first ranking
        for rank, idx in enumerate(rank_list1, start=1):
            scores[idx] += 1.0 / (k + rank)

        # Add scores from second ranking
        for rank, idx in enumerate(rank_list2, start=1):
            scores[idx] += 1.0 / (k + rank)

        return scores

In [5]:
documents = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Python is a popular programming language for data science",
    "Natural language processing enables computers to understand text",
    "Computer vision allows machines to interpret visual information"
]

system = HybridSearchSystem(documents)

Loading bi-encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [6]:
# Search
query = "How do machines learn from data?"
results = system.search(query, top_k=3, use_rerank=True)

for i, result in enumerate(results, 1):
    print(f"{i}. Score: {result['score']:.4f}")
    print(f"   {result['document']}\n")

1. Score: 0.8134
   Machine learning is a subset of artificial intelligence

2. Score: -4.2642
   Computer vision allows machines to interpret visual information

3. Score: -5.2855
   Natural language processing enables computers to understand text

